In [ ]:
import os
import sys

if os.getcwd().endswith("notebooks"):
    os.chdir("..")

sys.path.append(os.path.abspath("./"))

print(f"Current work directory: {os.getcwd()}")

In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata
import joblib
import scipy.sparse as sp
import h5py
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import scripts.plotting as pl
import scripts.utils as ut

In [ ]:
print("Loading model and predicting...")
model = joblib.load('./Data/model/Compact_cat_after_tunning.pkl')
df_input = pd.read_csv('./CSV/GSM2406677_tunicamycin_test.csv',index_col= False)

expected_features = model.feature_names_
rename_dict = {'ATP5J2': 'ATP5MF'}

df_input.rename(columns=rename_dict, inplace=True)
df_input_ordered = df_input[expected_features]

train_df = pd.read_csv('./CSV/train_df.csv', nrows=1)
all_features = [col for col in train_df.columns if col != 'target']
scaler = joblib.load('./Data/model/scaler.pkl')
indices = [all_features.index(f) for f in expected_features]
means = scaler.mean_[indices]
scales = scaler.scale_[indices]

df_input_scaled = (df_input_ordered - means) / scales
predictions_tm = model.predict(df_input_scaled)

In [ ]:
# Load existing data to plot against
test_df = pd.read_csv('./CSV/test_df.csv')
pred_test = model.predict(test_df)

f = h5py.File('./Data/h5ad/adata_oc43_removed_normalized.h5ad', 'r')
obs_names = f['obs']['_index']['values'][:].astype(str)
cond_codes = f['obs']['condition']['codes'][:]
cond_cats = f['obs']['condition']['categories']['values'][:].astype(str)
conditions = np.array([cond_cats[code] for code in cond_codes])
pct = f['obs']['pct_counts_oc43'][:]
tot = f['obs']['total_counts_oc43'][:]
y_h5ad = f['obs']['y_log1p_per10k'][:]
f.close()

obs = pd.DataFrame({'condition': conditions, 'pct_counts_oc43': pct, 'total_counts_oc43': tot, 'y_log1p_per10k': y_h5ad}, index=obs_names)

TEST_SIZE = 0.2
RANDOM_STATE = 4
idx_test = []
for cond, idx in obs.groupby('condition').indices.items():
    _, te = train_test_split(list(idx), test_size=TEST_SIZE, random_state=RANDOM_STATE)
    idx_test.extend(te)

test_obs_names = obs.index[idx_test]



p_hi = 10.0 
eps = 0.1 
t_hi = 10 
pct_test = obs.loc[test_obs_names, 'pct_counts_oc43']
tot_test = obs.loc[test_obs_names, 'total_counts_oc43']

no_mask   = ((pct_test <= eps) | (tot_test <= t_hi)).fillna(False)
high_mask_cond = ((pct_test >= p_hi) & (tot_test > t_hi)).fillna(False)

labels = np.select(
    [no_mask, high_mask_cond],
    ['No infection', 'High infection'],
    default='Low infection'
)

low_mask = (labels == 'Low infection')
high_mask = (labels == 'High infection')

pred_healthy = pred_test[no_mask]

pred_low = pred_test[low_mask]
pred_high = pred_test[high_mask]

In [ ]:
# Create combined dataframe
s_healthy = pd.Series(pred_healthy, name="Score").to_frame()
s_healthy['Condition'] = 'No infection'

s_low = pd.Series(pred_low, name="Score").to_frame()
s_low['Condition'] = 'OC43 Low Infection'

s_high = pd.Series(pred_high, name="Score").to_frame()
s_high['Condition'] = 'OC43 High Infection'

s_tm = pd.Series(predictions_tm, name="Score").to_frame()
s_tm['Condition'] = 'K562 Tunicamycin (Stress)'

df_combined = pd.concat([s_healthy, s_low, s_high, s_tm], ignore_index=True)
categories = ['No infection', 'OC43 Low Infection', 'OC43 High Infection', 'K562 Tunicamycin (Stress)']

df_combined['Condition'] = pd.Categorical(df_combined['Condition'], categories=categories, ordered=True)

In [ ]:
def cliffs_delta(lst1, lst2):
    m, n = len(lst1), len(lst2)
    matrix = np.zeros((m, n))
    for i in range(m):
        for j in range(n):
            if lst1[i] > lst2[j]: matrix[i, j] = 1
            elif lst1[i] < lst2[j]: matrix[i, j] = -1
    return np.sum(matrix) / (m * n)

# Tunicamycin Delta
u_stat_no_tm, p_val_no_tm = stats.mannwhitneyu(s_tm['Score'], s_healthy['Score'], alternative='two-sided')
delta_no_tm = cliffs_delta(s_tm['Score'].values, s_healthy['Score'].values)
print(f"--- Stress vs No Infection ---")
print(f"Mann-Whitney U: p-value = {p_val_no_tm:.2e}")
print(f"Cliff's Delta: {delta_no_tm:.4f}\n")

u_stat_low_tm, p_val_low_tm = stats.mannwhitneyu(s_tm['Score'], s_low['Score'], alternative='two-sided')
delta_low_tm = cliffs_delta(s_tm['Score'].values, s_low['Score'].values)
print(f"--- Stress vs Low Infection ---")
print(f"Mann-Whitney U: p-value = {p_val_low_tm:.2e}")
print(f"Cliff's Delta: {delta_low_tm:.4f}\n")

u_stat_high_tm, p_val_high_tm = stats.mannwhitneyu(s_tm['Score'], s_high['Score'], alternative='two-sided')
delta_high_tm = cliffs_delta(s_tm['Score'].values, s_high['Score'].values)
print(f"--- Stress vs High Infection ---")
print(f"Mann-Whitney U: p-value = {p_val_high_tm:.2e}")
print(f"Cliff's Delta: {delta_high_tm:.4f}\n")

In [ ]:
# Plotting
custom_palette = {
    'No infection': '#4a90e2', 
    'OC43 Low Infection': '#F5A623', 
    'OC43 High Infection': '#e94e77', 
    #'K562 CRISPRi (Stress)': '#9b59b6',
    'K562 Tunicamycin (Stress)': '#50e3c2'
}

# 1. Density Plot
fig1, ax1 = plt.subplots(figsize=(10, 6))
sns.kdeplot(data=df_combined, x="Score", hue="Condition", common_norm=False, 
            fill=True, alpha=0.3, linewidth=2, palette=custom_palette, ax=ax1)
ax1.set_title('Score Distribution by Cellular State', fontsize=14, fontweight='bold')
ax1.set_xlabel('Predicted Score', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_ylim(0, 1.75)
sns.move_legend(ax1, "upper right")

def format_p(p): return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

stats_text = (
    "    Statistics (vs Tunicamycin):\n"
    f"- No Infection: d = {delta_no_tm:.2f} ({format_p(p_val_no_tm)})\n"
    f"- Low Infection: d = {delta_low_tm:.2f} ({format_p(p_val_low_tm)})\n"
    f"- High Infection: d = {delta_high_tm:.2f} ({format_p(p_val_high_tm)})\n"
)
props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
ax1.text(0.98, 0.725, stats_text, transform=ax1.transAxes, fontsize=11,
        verticalalignment='top', horizontalalignment='right', multialignment='left', bbox=props)
plt.tight_layout()
plt.savefig('./Plot/Figure3C_Tm_v2.png', dpi=300, bbox_inches='tight')

# 2. Violin Plot
fig2, ax2 = plt.subplots(figsize=(8, 6))
sns.violinplot(data=df_combined, x="Condition", y="Score", palette=custom_palette, inner='quartile', ax=ax2)
ax2.set_title('Violin Plot of Predicted Scores', fontsize=14, fontweight='bold')
ax2.set_xlabel('Condition', fontsize=12)
ax2.set_ylabel('Predicted Score', fontsize=12)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig('./Plot/Figure3D_Tm_v2.png', dpi=300, bbox_inches='tight')

print("All processing and plotting completed successfully!")